# BCT Hackathon User Modelling 
> Version : 4
## Goal 
Build an agent that understands users deeply enough to simulate their reviews — capturing tone, rating behaviour, and contextual nuance.
- Simulate star ratings and written reviews for 
unseen items
- Leverage user history, item metadata, and 
contextual signals
- Evaluated on review quality, rating accuracy, 
and behavioural fidelity

### Notebook version 2 
This is the final version of the first build of the it entails downloading the dataset from hugging face (we used the beauty dataset first )
- then perfroming some much needed Exploratory Data Analysis on the data.
- then we created a function to clean the data into review rich data >10 or 100 words reviews
- then we used a hardcoded-review persona builder it builds persona on the reviewer using thier reviews
- then finally it uses gemini 2.5 to generate reviews
- then we then evaluate the data

## Notebook version 3 
- use a multi-agent workflow one agent gets the persona another builds the reviews
- we use pydantic to structure the output of the agents so the agents can give us what we want and not unwanted stuff
- check the evaluation data (hope its not leaking we should be spliting the reviews per users )
- try techniques to save tokens 
- also add something that limits character generation the review generated must be within the average character length of the reviewer (dont write more words than a reviewer will write)

## Notebook version 4 (Here now)
- adding rag to ground agent 2 reviews
- try cross-domain reviewing
- also adding this points
- also need to set up a way to define the products for the ai to understand what it is reviewing 
>Point 1 — Unseen item metadata missing from Agent 2
This one is completely valid and it's the same root cause behind your generation drift problem. Agent 2 currently receives the product title and a truncated description. That's not enough for it to write a product-aware review. It falls back to generic category patterns because it doesn't know enough about the specific item.
What Agent 2 actually needs is a structured item card — not just the title but the category, price tier, key features, brand, and any product-specific attributes. For a skincare product that means ingredients, skin type targeting, and claims. For electronics that means core specs. The item card should be built from your metadata join at data preparation time and passed to Agent 2 as a structured block, not a truncated description string.
This is also where your asin2category.json file from the Kaggle dataset becomes useful. You have a 35 million entry lookup table mapping every ASIN to its category. That's a rich source of item context you haven't used yet.

>Point 2 — Rating before text generation
This is the most technically important point of the three and it's currently broken in your system in a subtle way. You're asking Agent 2 to produce rating and review simultaneously in one JSON object. The LLM generates tokens left to right — if it writes the review text before the rating field, it generates whatever review sounds right and then assigns a rating to match. That's backwards.
The fix is to enforce field ordering in your JSON output so rating always appears before review. JSON objects don't have guaranteed key ordering in the spec but in practice LLMs follow the order they see in the example. Your concrete output example in Agent 2's prompt currently shows reasoning, then rating, then title, then review — which is actually already the right order. The problem is that Gemini doesn't always follow it.
The stronger fix is to split this into two sequential generation steps inside Agent 2. First generate the rating with a brief justification. Then generate the review conditioned on that rating. This forces the autoregressive generation to commit to a number and then write text that justifies it — not the reverse. It directly addresses the incoherence problem where positive text gets a 2-star rating.
This also improves your RMSE because the rating decision is now isolated and deliberate rather than being a byproduct of whatever text happened to get generated.

>Point 3 — Contextual signals injection
This one needs to be interpreted carefully against your actual brief. The hackathon brief says "contextual signals" but in the Amazon reviews context this means signals available in the review data itself — things like whether it was a verified purchase, the time gap between purchase and review, whether the user reviewed multiple items from the same brand recently, and their rating trajectory at the time of the review.
It does not mean real-time signals like time of day or device — those aren't in your dataset and you can't simulate them without inventing data, which would hurt your evaluation scores.
What you can inject from your existing data is genuinely useful contextual grounding. How long after their last review is this review being written — are they in an active reviewing period or returning after a long gap? Have they recently reviewed other products in the same category — do they have fresh category context? What is their rating trajectory at this point in time — are they in a generous phase or a critical phase based on their recent reviews? These are all derivable from your cleaned dataset and they add real signal to Agent 2's generation.

>What needs to change in your system
These three points map to three specific changes in priority order.
First, build a proper item card constructor that pulls from your metadata and asin2category lookup. This fixes the product awareness problem and should move your ROUGE-2 score meaningfully.
Second, restructure Agent 2 into two sequential calls — rating generation then review generation. This fixes the coherence problem and should improve RMSE consistency.
Third, add a contextual signals block to Agent 2's prompt that injects the user's reviewing context at the time of the holdout review — their recent rating trend, time since last review, and category familiarity score. This is derivable from your existing data with no additional API calls.
All three are buildable before your deadline and all three have direct mappings to scoring criteria — item metadata improves ROUGE, two-step generation improves RMSE, contextual signals improve behavioural fidelity. Each one earns you points in a different evaluation dimension.
Want to build all three now?

## importing dependencies

In [1]:
# ── INSTALL FIRST ────────────────────────────────────────────────
!pip install datasets
!pip install bert-score
!pip install rouge-score
!pip install faiss-cpu sentence-transformers

In [2]:
import json 
import re
import hashlib
import os
import faiss
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from google import genai
import google.generativeai as genai
from typing import Optional
from enum import Enum
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field, field_validator

from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


In [3]:
# use the kaggle secret enviroment to secure my api key 

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
HF_Key = UserSecretsClient().get_secret("HF_Key")

In [4]:


# ── DOWNLOAD ─────────────────────────────────────────────────────
!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz" \
    -O "/kaggle/working/All_Beauty_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz" \
    -O "/kaggle/working/meta_All_Beauty.jsonl.gz"

# confirm sizes — should be several MB each
!ls -lh /kaggle/working/*.gz

/kaggle/working/All 100%[===================>]  90.07M  43.7MB/s    in 2.1s    
/kaggle/working/met 100%[===================>]  38.02M  44.4MB/s    in 0.9s    
-rw-r--r-- 1 root root 91M Jan 16  2025 /kaggle/working/All_Beauty_reviews.jsonl.gz
-rw-r--r-- 1 root root 39M Jan 16  2025 /kaggle/working/meta_All_Beauty.jsonl.gz


In [5]:
import pandas as pd

# ── LOAD ─────────────────────────────────────────────────────────
print("Loading reviews...")
reviews_df = pd.read_json(
    '/kaggle/working/All_Beauty_reviews.jsonl.gz',
    lines       = True,
    compression = 'gzip'
)
print(f"Reviews : {reviews_df.shape}")
print(f"Columns : {reviews_df.columns.tolist()}")

print("\nLoading metadata...")
meta_df = pd.read_json(
    '/kaggle/working/meta_All_Beauty.jsonl.gz',
    lines       = True,
    compression = 'gzip'
)
print(f"Metadata : {meta_df.shape}")
print(f"Columns  : {meta_df.columns.tolist()}")

Loading reviews...
Reviews : (701528, 10)
Columns : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Loading metadata...
Metadata : (112590, 14)
Columns  : ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']


In [6]:
# ── JOIN ─────────────────────────────────────────────────────────
# flatten list fields in metadata
meta_df['description_text'] = meta_df['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) and x else ''
)
meta_df['features_text'] = meta_df['features'].apply(
    lambda x: ' | '.join(x[:3]) if isinstance(x, list) and x else ''
)

# slim metadata down
meta_slim = meta_df[[
    'parent_asin',
    'title',
    'description_text',
    'features_text',
    'price',
    'store',
    'main_category'
]].drop_duplicates(subset='parent_asin')

# join
df = reviews_df.merge(meta_slim, on='parent_asin', how='left')

# rename to match your pipeline
df = df.rename(columns={

    'title_x'          : 'review_title',
    'title_y'          : 'product_title',
})

print(f"\nJoined shape : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")


Joined shape : (701528, 16)
Columns      : ['rating', 'review_title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description_text', 'features_text', 'price', 'store', 'main_category']


In [7]:
# ── HEALTH CHECK ─────────────────────────────────────────────────
user_counts = df.groupby('user_id').size()
viable      = user_counts[user_counts >= 20]

print("=" * 50)
print("HEALTH CHECK")
print("=" * 50)
print(f"Total reviews         : {len(df):,}")
print(f"Unique users          : {df['user_id'].nunique():,}")
print(f"Unique products       : {df['asin'].nunique():,}")
print(f"Users with 20+ reviews: {len(viable):,}")
print(f"Avg review length     : {df['text'].str.split().str.len().mean():.0f} words")
print(f"Rating distribution   : {df['rating'].value_counts().sort_index().to_dict()}")
#print(f"Missing product title : {df['product_title'].isna().sum():,}")
#print(f"Missing description   : {df['description_text'].isna().sum():,}")
print(f"Verified purchase %   : {df['verified_purchase'].mean()*100:.1f}%")

HEALTH CHECK
Total reviews         : 701,528
Unique users          : 631,986
Unique products       : 115,709
Users with 20+ reviews: 117
Avg review length     : 33 words
Rating distribution   : {1: 102080, 2: 43034, 3: 56307, 4: 79381, 5: 420726}
Verified purchase %   : 90.5%


## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [8]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
duplicate_df = df.drop("images",axis = 1)
print(f"\nDuplicate rows: {duplicate_df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (701528, 16)

Column dtypes:
rating                        int64
review_title                 object
text                         object
images                       object
asin                         object
parent_asin                  object
user_id                      object
timestamp            datetime64[ns]
helpful_vote                  int64
verified_purchase              bool
product_title                object
description_text             object
features_text                object
price                       float64
store                        object
main_category                object
dtype: object

Missing values:
rating                    0
review_title              0
text                      0
images                    0
asin                      0
parent_asin               0
user_id                   0
timestamp                 0
helpful_vote              0
verified_purchase         0
product_title             0
description_text        

### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [9]:
## understanding the schema of the review dataset 
df.iloc[0].to_dict()

{'rating': 5,
 'review_title': 'Such a lovely scent but not overpowering.',
 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
 'images': [],
 'asin': 'B00YQ6X8EO',
 'parent_asin': 'B00YQ6X8EO',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': Timestamp('2020-05-05 14:08:48.923000'),
 'helpful_vote': 0,
 'verified_purchase': True,
 'product_title': 'Herbivore - Natural Sea Mist Texturizing Salt Spray (Coconut, 8 oz)',
 'description_text': 'If given the choice, weÕd leave most telltale signs of the beachÑsunburns, sandy toes, crab claw pinches, etc.Ñat the beach where they belong. The one thing wish we could take with us? That salty sea breeze. This all-natural spray manages, magically, to bottle the effect of sea mist s

In [10]:
#checking the columns in the data set 
df.columns.tolist()

['rating',
 'review_title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'product_title',
 'description_text',
 'features_text',
 'price',
 'store',
 'main_category']

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona

- the results here are quite worse to build the prototype i will use users with 5 or more reviews 

In [11]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

count    631986.000000
mean          1.110037
std           0.753202
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         165.000000
dtype: float64

In [12]:
# How many viable user?
five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")


The number of users with five reviews or more are 1620,
        The number of users with ten reviews or more are 330,
        while the number of users with twenty reviews or more are 117


In [13]:
# Distribution shape
user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

1     583553
2      39274
3       5713
4       1826
5        558
6        351
7        181
8        130
9         70
11        35
10        35
13        33
12        26
16        20
15        20
14        19
17        14
21         9
26         8
24         8
Name: count, dtype: int64

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [14]:
# Global rating distribution
df['rating'].value_counts().sort_index()

rating
1    102080
2     43034
3     56307
4     79381
5    420726
Name: count, dtype: int64

In [15]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

count    631986.000000
mean          3.948681
std           1.487903
min           1.000000
25%           3.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [16]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

count    48433.000000
mean         0.697705
std          0.918379
min          0.000000
25%          0.000000
50%          0.000000
75%          1.414214
max          2.828427
Name: rating, dtype: float64

In [17]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

count    1.620000e+03
mean    -1.484943e-02
std      2.539332e-01
min     -1.200000e+00
25%     -8.571429e-02
50%      2.013140e-16
75%      5.714286e-02
max      1.100000e+00
Name: rating_slope, dtype: float64
trend
insufficient_data        630366
consistent                  732
increasingly_critical       477
increasingly_generous       411
Name: count, dtype: int64


In [18]:
df.head()

,rating,review_title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq
72840,5,Five Stars,Great product....excellent price for good resu...,[],B013HR1A92,B013HR1A92,AE222BBOVZIF42YOOPNBXL4UUMYA,2016-03-10 00:27:52.000,0,True,UNGLINGA Black Mask Blackhead Remover Purifyin...,"CONCERNS: Enlarged pore, Blackheads, Anti-agin...",DEEP CLEANSING BLACKHEAD REMOVER WITH TOOL - O...,NaN,UNGLINGA,All Beauty,1
392969,5,Nice consistency and great smell,[[VIDEOID:8b3489ae8e6301ceb95a2973d7f721f3]],[],B0BTT658PQ,B0BTT658PQ,AE222FP7YRNFCEQ2W3ZDIGMSYTLQ,2023-03-07 03:10:12.143,0,True,Hair Growth Serum for Women and Men – 100% Mad...,,,NaN,Svvimer,All Beauty,1
636591,5,Wow,It tastes good,[],B00PBDMRES,B00PBDMRES,AE222X475JC6ONXMIKZDFGQ7IAUA,2017-01-06 18:55:24.000,2,True,1775 Valor For Men 3.4 oz EDT Spray By Royal C...,"The Modern Maverick""The true essence of streng...",,NaN,1775 VALOR,All Beauty,1
293687,4,Lensoclean Unit,The cleaning unit does a good job of cleaning ...,[],B00012FPSO,B00012FPSO,AE222Y4WTST6BUZ4J5Y2H6QMBITQ,2013-06-24 21:11:42.000,1,True,drtulz ULTRASONIC CLEANING,The product has EUROPLUG. For US market needs ...,1111,NaN,drtulz,All Beauty,1
589847,5,Sus colores,Son como en la foto,[],B07QNPXBLH,B07QNPXBLH,AE2232TEZOEWQLAFEX2NA6VBGMYQ,2019-07-30 05:47:26.197,0,True,SIQUK 12 Pieces Headbands with Twist Knot Head...,,,NaN,SIQUK,All Beauty,1


In [19]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

"                 timestamp  rating  review_seq                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             text\n242833 2016-07-04 20:3

In [20]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n📊 QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f} ⭐  (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [21]:

"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

USER PROFILE: AHBWH2LBU3NFLD46GKJKIBAHKXEQ

📊 QUICK STATS
   Total reviews   : 39
   Avg rating      : 4.05 ⭐  (std: 1.00)
   Avg review length: 104 words
   Rating breakdown: {1: 1, 2: 3, 3: 3, 4: 18, 5: 14}
   Rating trend    : ↑ more generous over time  (early avg: 3.89 → late avg: 4.20)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Aug 2020  |  ⭐⭐⭐⭐ (4/5)  |  237 words
     I try to avoid using traditional files and emery boards
     on my nails. I take a lot of medications that make my
     nails weak and brittle, and any surface that's too
     abrasive wreaks havoc on my fingertips. I can't carry a
     full size glass nail file with me everywhere, so I was
     on the hunt for a travel sized file that could fit in
     my pocket, wallet, or purse. When these came up, I
     thought I'd give them a shot. The pros are the files
     are a great size, they're packaged well, and I see them
     lasting for awhile. 

In [22]:
# random user with 30+ reviews
user_history = profile_user(df)

Randomly selected user: AGZUJTI7A3JFKB4FP5JOH6NVAJIQ_1

USER PROFILE: AGZUJTI7A3JFKB4FP5JOH6NVAJIQ_1

📊 QUICK STATS
   Total reviews   : 87
   Avg rating      : 4.72 ⭐  (std: 0.52)
   Avg review length: 33 words
   Rating breakdown: {2: 1, 4: 21, 5: 65}
   Rating trend    : → consistent rater  (early avg: 4.65 → late avg: 4.80)

📝 FULL REVIEW HISTORY  (oldest → newest)
-----------------------------------------------------------------

[1]  Sep 2020  |  ⭐⭐⭐⭐⭐ (5/5)  |  11 words
     Hanks you I love it so much and smell so good


[2]  Sep 2020  |  ⭐⭐⭐⭐ (4/5)  |  13 words
     Super cute, my kids love it!, but its too small for my
     side


[3]  Sep 2020  |  ⭐⭐⭐⭐⭐ (5/5)  |  10 words
     I love it so cute and match with my skin


[4]  Oct 2020  |  ⭐⭐⭐⭐⭐ (5/5)  |  27 words
     I love the quality and quantity for price, you can get
     this in the beauty supplies for the double price and
     not as good as this one.


[5]  Oct 2020  |  ⭐⭐⭐⭐⭐ (5/5)  |  25 words
     My husband uses the

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

In [23]:
# Review length distribution
df['text'].str.split().str.len().describe()

count    701528.000000
mean         32.750720
std          45.973273
min           0.000000
25%           8.000000
50%          19.000000
75%          40.000000
max        2585.000000
Name: text, dtype: float64

In [24]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

74841

In [25]:
# Avg text length per user (computationaly expensive to load)
#avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [26]:
#display(avg_text_per_user)

In [27]:
#print(avg_text_per_user.value_counts())


In [28]:
# Verified purchase flag

df['verified_purchase'].value_counts()
# prefer verified = True

verified_purchase
True     634969
False     66559
Name: count, dtype: int64

## Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data 

In [29]:


def clean_amazon_reviews(df, 
                          min_reviews=20, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop("images", axis = 1)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[ user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining      user_counts         : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [30]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

AMAZON REVIEWS — CLEANING PIPELINE

▶ Starting shape: 701,528 rows × 17 columns

[1] Duplicate rows removed   : 0
    Remaining                : 701,528

[2] Duplicate columns removed: 0
    Remaining columns        : ['rating', 'review_title', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'product_title', 'description_text', 'features_text', 'price', 'store', 'main_category', 'review_seq']

[3] Rows dropped (missing critical fields): 0
    Remaining                              : 701,528

[4] Rows dropped (empty / short reviews) : 456,668
    Min word count threshold             : 30 words
    Remaining                            : 244,860

[5] Rows dropped (unverified purchases)  : 41,244
    Remaining                            : 203,616

[6] Rows dropped (invalid ratings)       : 0
    Remaining                            : 203,616

[7] Users dropped (< 10 reviews)          : 192,293
    Viable users remaining      user_counts         :

In [31]:
rich_df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq,word_count
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,"Flawless Finish Foundation, Colour Changing Fo...",,,NaN,CIDBEST,All Beauty,1,158
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,"Flawless Liquid Foundation Cream, Liquid Found...",,,NaN,Cherioll,All Beauty,2,97
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,"Rapid Reduction Eye Cream, Under Eye Cream, Un...",,,NaN,CIDBEST,All Beauty,3,137
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,"Concealer Cream,Makeup concealer,Skin Lighteni...",,,NaN,SCOBUTY,All Beauty,4,132
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,W-Airfit Primer Face Makeup Base Pink Isolatio...,,,NaN,Lofu,All Beauty,5,124


In [32]:
rich_df.to_csv('rich_users.csv', index=False)

## Split the cleaned data 
 this function splits the cleaned data to the training data(persona_df), the validation data (val_df) and the test data 

In [33]:
import pandas as pd
import numpy as np

def split_user_reviews(clean_df, min_reviews=20, n_test=1, n_val=1):
    """
    Splits cleaned Amazon reviews into three sets per user:
    - Persona set   : bulk history for Agent 1 profiling
    - Validation set: second-to-last reviews for prompt tuning
    - Test set      : last reviews for final evaluation only

    Parameters:
        clean_df    : your cleaned dataframe from clean_amazon_reviews()
        min_reviews : minimum reviews a user needs to be included
        n_test      : number of reviews to hold out for test (default 1)
        n_val       : number of reviews to hold out for validation (default 1)

    Returns:
        persona_df  : DataFrame — bulk user history
        val_df      : DataFrame — validation reviews
        test_df     : DataFrame — test reviews (lock this away)
        split_stats : dict — summary of the split
    """

    print("=" * 60)
    print("TEMPORAL REVIEW SPLIT")
    print("=" * 60)
    print(f"  Min reviews required : {min_reviews}")
    print(f"  Test holdout         : last {n_test} review(s) per user")
    print(f"  Validation holdout   : {n_val} review(s) before test")
    print(f"  Persona minimum      : {min_reviews - n_test - n_val} reviews\n")

    # ── VALIDATE INPUTS ──────────────────────────────────────────
    required_cols = ['user_id', 'timestamp', 'text', 'rating']
    missing       = [c for c in required_cols if c not in clean_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    min_needed = min_reviews + n_test + n_val
    
    # ── SORT BY USER + TIME ──────────────────────────────────────
    df = clean_df.sort_values(
        ['user_id', 'timestamp']
    ).reset_index(drop=True)

    # ── SPLIT PER USER ───────────────────────────────────────────
    persona_rows  = []
    val_rows      = []
    test_rows     = []
    skipped_users = []

    user_groups = df.groupby('user_id')

    for user_id, group in user_groups:
        group = group.reset_index(drop=True)
        n     = len(group)

        # skip users without enough reviews
        if n < min_needed:
            skipped_users.append({
                'user_id'      : user_id,
                'review_count' : n,
                'reason'       : f"needs {min_needed}, has {n}"
            })
            continue

        # temporal split — oldest to newest
        test_slice    = group.iloc[-n_test:]                      # last n_test
        val_slice     = group.iloc[-(n_test + n_val):-n_test]     # before test
        persona_slice = group.iloc[:-(n_test + n_val)]            # everything before

        # tag each split
        test_slice    = test_slice.copy()
        val_slice     = val_slice.copy()
        persona_slice = persona_slice.copy()

        test_slice['split']    = 'test'
        val_slice['split']     = 'validation'
        persona_slice['split'] = 'persona'

        test_rows.append(test_slice)
        val_rows.append(val_slice)
        persona_rows.append(persona_slice)

    # ── BUILD DATAFRAMES ─────────────────────────────────────────
    persona_df = pd.concat(persona_rows,  ignore_index=True) if persona_rows else pd.DataFrame()
    val_df     = pd.concat(val_rows,      ignore_index=True) if val_rows     else pd.DataFrame()
    test_df    = pd.concat(test_rows,     ignore_index=True) if test_rows    else pd.DataFrame()

    # ── SANITY CHECKS ────────────────────────────────────────────
    # 1 — no user should appear in both val and test with the same review
    persona_ids = set(persona_df['user_id'].unique()) if len(persona_df) else set()
    val_ids     = set(val_df['user_id'].unique())     if len(val_df)     else set()
    test_ids    = set(test_df['user_id'].unique())    if len(test_df)    else set()

    # every test user must have a persona
    missing_persona = test_ids - persona_ids
    if missing_persona:
        print(f"⚠️  WARNING: {len(missing_persona)} test users have no persona history")

    # check for timestamp ordering — test must always be after persona
    ordering_violations = 0
    for user_id in list(test_ids)[:50]:    # sample check on first 50 users
        p_times = persona_df[persona_df['user_id'] == user_id]['timestamp']
        t_times = test_df[test_df['user_id'] == user_id]['timestamp']
        if len(p_times) and len(t_times):
            if p_times.max() >= t_times.min():
                ordering_violations += 1

    if ordering_violations > 0:
        print(f"⚠️  WARNING: {ordering_violations} users have timestamp ordering issues")
    else:
        print(f"✅ Timestamp ordering verified — test is always after persona")

    # ── SPLIT STATS ──────────────────────────────────────────────
    split_stats = {
        'total_users_before' : df['user_id'].nunique(),
        'users_kept'         : len(test_ids),
        'users_skipped'      : len(skipped_users),
        'persona_reviews'    : len(persona_df),
        'val_reviews'        : len(val_df),
        'test_reviews'       : len(test_df),
        'avg_persona_len'    : round(
            persona_df.groupby('user_id').size().mean(), 1
        ) if len(persona_df) else 0,
        'skipped_users'      : skipped_users
    }

    # ── PRINT SUMMARY ────────────────────────────────────────────
    print(f"{'=' * 60}")
    print(f"SPLIT SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Total users in dataset : {split_stats['total_users_before']:,}")
    print(f"  Users kept             : {split_stats['users_kept']:,}")
    print(f"  Users skipped          : {split_stats['users_skipped']:,}")
    print(f"\n  Persona set            : {split_stats['persona_reviews']:,} reviews")
    print(f"  Validation set         : {split_stats['val_reviews']:,} reviews")
    print(f"  Test set               : {split_stats['test_reviews']:,} reviews")
    print(f"\n  Avg persona length     : {split_stats['avg_persona_len']} reviews/user")
    print(f"{'=' * 60}")

    # ── WARN ABOUT TEST SET ──────────────────────────────────────
    print(f"\n🔒 TEST SET LOCKED — {len(test_df)} reviews")
    print(f"   Do NOT use test_df for prompt tuning.")
    print(f"   Use val_df during development.")
    print(f"   Run test_df ONCE at final evaluation only.\n")

    return persona_df, val_df, test_df, split_stats

In [34]:
# ── RUN IT ───────────────────────────────────────────────────────

persona_df, val_df, test_df, split_stats = split_user_reviews(
    clean_df    = rich_df,
    min_reviews = 8,
    n_test      = 1,
    n_val       = 1
)

# ── SPOT CHECK ONE USER ──────────────────────────────────────────
# verify the split looks right for a real user

sample_user = test_df['user_id'].iloc[0]

print(f"SPOT CHECK — User: {sample_user}")
print(f"{'─' * 50}")

p = persona_df[persona_df['user_id'] == sample_user]
v = val_df[val_df['user_id'] == sample_user]
t = test_df[test_df['user_id'] == sample_user]

print(f"Persona  : {len(p)} reviews | "
      f"last timestamp: {p['timestamp'].max()}")
print(f"Val      : {len(v)} reviews | "
      f"timestamp: {v['timestamp'].values}")
print(f"Test     : {len(t)} reviews | "
      f"timestamp: {t['timestamp'].values}")

# confirm ordering
assert p['timestamp'].max() < v['timestamp'].min(), \
    "❌ Persona bleeds into validation"
assert v['timestamp'].max() < t['timestamp'].min(), \
    "❌ Validation bleeds into test"

print(f"\n✅ Ordering confirmed — persona < validation < test")
print(f"\nSample persona review  : {p.iloc[-1]['text'][:100]}...")
print(f"Sample val review      : {v.iloc[0]['text'][:100]}...")
print(f"Sample test review     : {t.iloc[0]['text'][:100]}...")

TEMPORAL REVIEW SPLIT
  Min reviews required : 8
  Test holdout         : last 1 review(s) per user
  Validation holdout   : 1 review(s) before test
  Persona minimum      : 6 reviews

✅ Timestamp ordering verified — test is always after persona
SPLIT SUMMARY
  Total users in dataset : 7
  Users kept             : 7
  Users skipped          : 0

  Persona set            : 86 reviews
  Validation set         : 7 reviews
  Test set               : 7 reviews

  Avg persona length     : 12.3 reviews/user

🔒 TEST SET LOCKED — 7 reviews
   Do NOT use test_df for prompt tuning.
   Use val_df during development.
   Run test_df ONCE at final evaluation only.

SPOT CHECK — User: AE7P3G7DWP3VVFKOK2H2PPTK2TOA
──────────────────────────────────────────────────
Persona  : 9 reviews | last timestamp: 2020-07-28 15:46:53.679000
Val      : 1 reviews | timestamp: ['2020-09-07T19:59:36.164000000']
Test     : 1 reviews | timestamp: ['2020-10-20T17:10:00.129000000']

✅ Ordering confirmed — persona < valida

# Part 2 Build a Persona 


In [35]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional
from enum import Enum


# ── HELPER — reusable truncation logic ───────────────────────────

def truncate(value: str, limit: int) -> str:
    """Truncate string to limit, appending ellipsis if cut."""
    if isinstance(value, str) and len(value) > limit:
        return value[:limit - 3] + "..."
    return value


# ── ENUMS — unchanged ─────────────────────────────────────────────

class RatingTendency(str, Enum):
    generous  = "generous"
    harsh     = "harsh"
    balanced  = "balanced"

class Consistency(str, Enum):
    consistent = "consistent"
    variable   = "variable"
    polarised  = "polarised"

class WritingStyle(str, Enum):
    verbose  = "verbose"
    moderate = "moderate"
    terse    = "terse"

class PriceSensitivity(str, Enum):
    very_high = "very high"
    high      = "high"
    moderate  = "moderate"
    low       = "low"

class SkepticismLevel(str, Enum):
    very_high = "very high"
    high      = "high"
    moderate  = "moderate"
    low       = "low"


# ── RATING BEHAVIOUR ──────────────────────────────────────────────

class RatingBehaviour(BaseModel):
    tendency   : RatingTendency
    consistency: Consistency
    never_gives: list[int] = Field(
        default_factory = list,
        description     = "STRICT: List only integers 1-5. e.g. [5] or []"
    )
    pattern: str = Field(
        max_length  = 200,
        description = "STRICT CONSTRAINT: Under 200 characters. Use fragments if needed."
    )

    @field_validator('pattern', mode='before')
    @classmethod
    def truncate_pattern(cls, v):
        return truncate(str(v), 200) if v else v

    @field_validator('never_gives', mode='before')
    @classmethod
    def validate_never_gives(cls, v):
        if not isinstance(v, list):
            return []
        return [int(x) for x in v if str(x).strip().isdigit() and 1 <= int(x) <= 5]


# ── WRITING VOICE ─────────────────────────────────────────────────

class WritingVoice(BaseModel):
    style    : WritingStyle
    tone     : str = Field(
        max_length  = 150,
        description = "STRICT CONSTRAINT: Under 150 characters. Single descriptive phrase."
    )
    structure: str = Field(
        max_length  = 250,
        description = "STRICT CONSTRAINT: Under 250 characters. How they organise a review."
    )
    signature_phrases: list[str] = Field(
        default_factory = list,
        description     = "Actual phrases from their reviews. No invented phrases."
    )

    @field_validator('tone', mode='before')
    @classmethod
    def truncate_tone(cls, v):
        return truncate(str(v), 150) if v else v

    @field_validator('structure', mode='before')
    @classmethod
    def truncate_structure(cls, v):
        return truncate(str(v), 250) if v else v

    @field_validator('signature_phrases', mode='before')
    @classmethod
    def clean_phrases(cls, v):
        if not isinstance(v, list):
            return []
        # truncate any individual phrase that is too long
        return [str(p)[:100] for p in v if p]


# ── NIGERIAN SIGNALS ──────────────────────────────────────────────

class NigerianSignals(BaseModel):
    price_sensitivity  : PriceSensitivity
    scepticism         : SkepticismLevel
    community_oriented : bool
    cultural_notes     : Optional[str] = Field(
        default     = None,
        max_length  = 1000,
        description = "STRICT CONSTRAINT: Under 1000 characters. Nigerian consumer signals observed."
    )

    @field_validator('cultural_notes', mode='before')
    @classmethod
    def truncate_cultural_notes(cls, v):
        if v is None:
            return v
        return truncate(str(v), 1000)

    @field_validator('community_oriented', mode='before')
    @classmethod
    def coerce_bool(cls, v):
        # handle cases where LLM returns "true"/"false" as strings
        if isinstance(v, str):
            return v.lower() in ('true', 'yes', '1')
        return bool(v)


# ── PERSONA DOSSIER ───────────────────────────────────────────────

class PersonaDossier(BaseModel):
    """
    Agent 1 output — the inter-agent contract.
    Structural fields validated strictly.
    Discovery fields free-form with graceful truncation.
    """
    user_id      : str
    core_identity: str = Field(
        max_length  = 600,
        description = "STRICT CONSTRAINT: Under 600 characters. Who this person is as a reviewer."
    )
    rating_behaviour    : RatingBehaviour
    writing_voice       : WritingVoice
    deep_traits         : list[str] = Field(
        min_length  = 3,
        description = "Specific discovered traits. Each must be a full observation, not a single word."
    )
    what_they_care_about: list[str] = Field(
        min_length  = 1,
        description = "Ranked by importance — most dominant first."
    )
    what_they_ignore    : list[str] = Field(default_factory=list)
    context_clues       : Optional[str] = Field(
        default     = None,
        max_length  = 1000,
        description = "STRICT CONSTRAINT: Under 1000 characters. Life context revealed in reviews."
    )
    nigerian_signals    : NigerianSignals
    simulation_brief    : str = Field(
        max_length  = 1000,
        description = (
            "STRICT CONSTRAINT: Under 1000 characters. "
            "Written directly to the generation model. "
            "How to sound like this person — voice, priorities, quirks."
        )
    )

    # computed fields — injected after LLM call
    avg_rating  : float = 3.0
    rating_std  : float = 0.0
    review_count: int   = 0

    # ── TRUNCATION VALIDATORS ─────────────────────────────────────

    @field_validator('core_identity', mode='before')
    @classmethod
    def truncate_core_identity(cls, v):
        return truncate(str(v), 600) if v else v

    @field_validator('context_clues', mode='before')
    @classmethod
    def truncate_context_clues(cls, v):
        if v is None:
            return v
        return truncate(str(v), 1000)

    @field_validator('simulation_brief', mode='before')
    @classmethod
    def truncate_simulation_brief(cls, v):
        return truncate(str(v), 1000) if v else v

    @field_validator('deep_traits', mode='before')
    @classmethod
    def clean_deep_traits(cls, v):
        if not isinstance(v, list):
            return []
        # truncate any individual trait that is excessively long
        # but don't filter content — agent decides what's relevant
        return [str(t)[:300] for t in v if t and len(str(t).strip()) > 3]

    @field_validator('what_they_care_about', mode='before')
    @classmethod
    def clean_cares_about(cls, v):
        if not isinstance(v, list):
            return ['product quality']
        return [str(t)[:200] for t in v if t]

    @field_validator('what_they_ignore', mode='before')
    @classmethod
    def clean_ignores(cls, v):
        if not isinstance(v, list):
            return []
        return [str(t)[:200] for t in v if t]

    # ── STRUCTURAL VALIDATORS — kept strict ───────────────────────

    @field_validator('avg_rating')
    @classmethod
    def valid_rating_range(cls, v):
        if not 1.0 <= v <= 5.0:
            raise ValueError(f"avg_rating {v} out of range 1–5")
        return round(v, 2)

    @field_validator('rating_behaviour')
    @classmethod
    def valid_never_gives(cls, v):
        for star in v.never_gives:
            if star not in [1, 2, 3, 4, 5]:
                raise ValueError(f"never_gives contains invalid value: {star}")
        return v

    # ── MODEL VALIDATOR — cross-field check ───────────────────────

    @model_validator(mode='after')
    def brief_references_identity(self):
        """
        Soft check — simulation_brief should not be empty
        if we have enough identity information to fill it.
        Falls back gracefully rather than raising.
        """
        if not self.simulation_brief or len(self.simulation_brief) < 20:
            self.simulation_brief = (
                f"Write as a {self.rating_behaviour.tendency} rater "
                f"with a {self.writing_voice.tone} tone. "
                f"They care most about: {', '.join(self.what_they_care_about[:2])}."
            )
        return self


# ── SIMULATED REVIEW ──────────────────────────────────────────────

class SimulatedReview(BaseModel):
    """
    Agent 2 output — the final deliverable.
    """
    user_id  : str
    asin     : str
    reasoning: str = Field(
        max_length  = 800,
        description = "STRICT CONSTRAINT: Under 800 characters."
    )
    rating  : int  = Field(ge=1, le=5)
    title   : str  = Field(max_length=200)
    review  : str  = Field(
        min_length  = 10,
        description = "Full review in the user's voice."
    )
    language: str = "english"

    @field_validator('reasoning', mode='before')
    @classmethod
    def truncate_reasoning(cls, v):
        return truncate(str(v), 800) if v else v

    @field_validator('title', mode='before')
    @classmethod
    def truncate_title(cls, v):
        return truncate(str(v), 200) if v else v

    @field_validator('rating', mode='before')
    @classmethod
    def coerce_and_clamp_rating(cls, v):
        # handle "4 stars", "4/5", "4.5" etc
        if isinstance(v, str):
            match = re.search(r'\d+', v)
            v = int(match.group()) if match else 3
        return max(1, min(5, int(float(v))))

    @field_validator('review', mode='before')
    @classmethod
    def review_not_placeholder(cls, v):
        placeholders = {'n/a', 'none', 'null', '[review]', 'review text', 'placeholder'}
        if not v or str(v).lower().strip() in placeholders:
            raise ValueError("Review is a placeholder — generation failed")
        return str(v)

In [36]:
# instantiating the Gemini Api 
genai.configure(api_key = GOOGLE_API_KEY)

# the client behaviour is the current one so we have to update our model taking this into consideration 
#client = genai.client(api_key = GOOGLE_API_KEY)

flash_model = genai.GenerativeModel("gemini-2.5-flash")
pro_model = genai.GenerativeModel("gemini-3.1-flash-lite")

In [37]:
# ── CACHE LAYER ──────────────────────────────────────────────────

class PersonaCache:
    """
    Stores Agent 1 outputs to disk.
    Same user never costs an API call twice.
    """
    def __init__(self, cache_dir='/kaggle/working/persona_cache'):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def _key(self, user_id):
        return os.path.join(
            self.cache_dir,
            f"{hashlib.md5(user_id.encode()).hexdigest()}.json"
        )

    def get(self, user_id):
        path = self._key(user_id)
        if os.path.exists(path):
            with open(path, 'r') as f:
                return json.load(f)
        return None

    def set(self, user_id, profile):
        with open(self._key(user_id), 'w') as f:
            json.dump(profile, f, indent=2)

    def exists(self, user_id):
        return os.path.exists(self._key(user_id))

    def size(self):
        return len(os.listdir(self.cache_dir))


cache = PersonaCache()


# ── TOKEN SAVING UTILITIES ───────────────────────────────────────

def compress_review(text, max_words=80):
    """
    Trims a review to max_words while keeping meaning.
    Removes excessive punctuation and whitespace.
    Saves tokens without losing signal.
    """
    if not text:
        return ""
    # collapse whitespace
    text = re.sub(r'\s+', ' ', str(text).strip())
    # remove repeated punctuation
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[.]{2,}', '...', text)
    words = text.split()
    if len(words) <= max_words:
        return text
    # truncate but end at a sentence boundary if possible
    truncated = ' '.join(words[:max_words])
    last_stop = max(
        truncated.rfind('.'),
        truncated.rfind('!'),
        truncated.rfind('?')
    )
    if last_stop > max_words * 3:   # only use boundary if it's not too short
        return truncated[:last_stop + 1]
    return truncated + '...'


def select_representative_reviews(texts, ratings, n=6):
    """
    Picks the most signal-rich reviews to send to Agent 1.
    Selects across the rating spectrum — not just recent ones.
    This gives Agent 1 a balanced picture in fewer tokens.

    Strategy:
    - 1 lowest rated review  (reveals complaints)
    - 1 highest rated review (reveals praise style)  
    - 1 middle rated review  (reveals nuance)
    - 3 most recent reviews  (reveals current voice)
    """
    if not texts or len(texts) == 0:
        return []

    paired    = list(zip(texts, ratings))
    selected  = []
    seen_idx  = set()

    # lowest rating
    min_idx = min(range(len(paired)), key=lambda i: paired[i][1])
    selected.append(paired[min_idx])
    seen_idx.add(min_idx)

    # highest rating
    max_idx = max(range(len(paired)), key=lambda i: paired[i][1])
    if max_idx not in seen_idx:
        selected.append(paired[max_idx])
        seen_idx.add(max_idx)

    # middle rating (closest to 3)
    mid_idx = min(
        [i for i in range(len(paired)) if i not in seen_idx],
        key=lambda i: abs(paired[i][1] - 3),
        default=None
    )
    if mid_idx is not None:
        selected.append(paired[mid_idx])
        seen_idx.add(mid_idx)

    # most recent (last n reviews, skip already selected)
    for i in range(len(paired) - 1, -1, -1):
        if i not in seen_idx and len(selected) < n:
            selected.append(paired[i])
            seen_idx.add(i)

    return selected


def build_compact_history(texts, ratings, max_words_per_review=80, n_reviews=6):
    """
    Builds a token-efficient review history block for Agent 1.
    Selects representative reviews and compresses each one.
    """
    selected = select_representative_reviews(texts, ratings, n=n_reviews)
    lines    = []
    for i, (text, rating) in enumerate(selected):
        compressed = compress_review(text, max_words=max_words_per_review)
        lines.append(f"[{rating}★] {compressed}")
    return "\n".join(lines)


# ── AGENT 1 WITH PYDANTIC ────────────────────────────────────────

def run_agent1_analyst(
    user_id    : str,
    texts      : list,
    ratings    : list,
    flash_model,
    max_retries: int = 2
) -> PersonaDossier:
    """
    Calls Gemini Flash and validates output against PersonaDossier schema.
    Retries once on validation failure before falling back.
    """

    # cache check
    cached = cache.get(user_id)
    if cached:
        try:
            return PersonaDossier(**cached)
        except Exception:
            pass   # cache is stale — re-run

    # compute stats
    avg_rating   = round(sum(ratings) / len(ratings), 2) if ratings else 3.0
    rating_std   = round(
        (sum((r - avg_rating)**2 for r in ratings) / len(ratings))**0.5, 2
    ) if ratings else 0.0
    history_block = build_compact_history(texts, ratings)
    stats_block   = (
        f"Total: {len(ratings)} reviews | "
        f"Avg: {avg_rating}/5 | "
        f"Std: {rating_std} | "
        f"Spread: {sorted(set(ratings))}"
    )

    # schema hint — show LLM the exact structure expected
    schema_hint = json.dumps(PersonaDossier.model_json_schema(), indent=2)

    prompt = f"""Analyse this Amazon reviewer and produce their persona dossier.
Discover traits dynamically — do not use generic categories.

"IMPORTANT: Be extremely concise. Use fragments not full sentences where possible. "
"Never exceed the character limits in the schema. Front-load the most important information."

STATS: {stats_block}

REVIEWS:
{history_block}

Return ONLY valid JSON matching this exact schema:
{schema_hint}

Rules:
- deep_traits must be specific observations, not generic words
- simulation_brief is written directly to the generation model
- never_gives is a list of integers e.g. [5] or []
- No markdown, no explanation, JSON only"""

    for attempt in range(max_retries + 1):
        try:
            response = flash_model.generate_content(prompt)
            raw      = response.text.strip()
            raw      = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
            raw      = re.sub(r'^```\s*',     '', raw, flags=re.MULTILINE)
            raw      = re.sub(r'\s*```$',     '', raw, flags=re.MULTILINE)

            data = json.loads(raw)

            # inject computed stats
            data['user_id']      = user_id
            data['avg_rating']   = avg_rating
            data['rating_std']   = rating_std
            data['review_count'] = len(ratings)

            # validate with Pydantic
            dossier = PersonaDossier(**data)

            # cache validated object
            cache.set(user_id, dossier.model_dump())
            return dossier

        except json.JSONDecodeError as e:
            print(f"  ⚠️  Attempt {attempt+1} — JSON parse failed: {e}")
        except ValueError as e:
            print(f"  ⚠️  Attempt {attempt+1} — Validation failed: {e}")
        except Exception as e:
            print(f"  ⚠️  Attempt {attempt+1} — Unexpected error: {e}")

    # fallback — minimal valid dossier
    print(f"  ❌ All attempts failed for {user_id} — using fallback")
    return PersonaDossier(
        user_id          = user_id,
        core_identity    = "Reviewer with insufficient profile data.",
        rating_behaviour = RatingBehaviour(
            tendency    = RatingTendency.balanced,
            consistency = Consistency.consistent,
            never_gives = [],
            pattern     = "No pattern detected."
        ),
        writing_voice = WritingVoice(
            style             = WritingStyle.moderate,
            tone              = "neutral",
            structure         = "standard",
            signature_phrases = []
        ),
        deep_traits          = [
            "No distinctive traits detected from available history",
            "Review history too sparse for deep profiling",
            "Defaults to standard reviewer behaviour",
            "No strong category preferences detected"
        ],
        what_they_care_about = ["product quality", "value for money"],
        what_they_ignore     = [],
        nigerian_signals     = NigerianSignals(
            price_sensitivity  = PriceSensitivity.moderate,
            scepticism         = SkepticismLevel.moderate,
            community_oriented = False,
            cultural_notes     = None
        ),
        simulation_brief = (
            f"Write a balanced review consistent with a {avg_rating:.1f}-star "
            f"average rater. Keep tone neutral and length moderate."
        ),
        avg_rating   = avg_rating,
        rating_std   = rating_std,
        review_count = len(ratings)
    )


In [38]:
# ── AGENT 2 WITH PYDANTIC and rag  ────────────────────────────────────────
def run_agent2_with_rag(
    dossier          : 'PersonaDossier',
    item_asin        : str,
    item_title       : str,
    item_description : str,
    item_category    : str,
    pro_model,
    index            : faiss.Index,
    metadata         : list,
    embedding_model  : SentenceTransformer,
    nigerian_language: Optional[str] = None,
    top_k            : int = 4,
    max_retries      : int = 2
) -> 'SimulatedReview':
    """
    Agent 2 with RAG grounding.

    Retrieves this user's most relevant past reviews for the
    target product, then injects them as evidence into the prompt.
    Agent 2 now has both:
        - Agent 1's analytical brief  (who they are)
        - RAG examples                (how they actually write)

    Parameters:
        dossier           : validated PersonaDossier from Agent 1
        item_asin         : product ASIN
        item_title        : product title
        item_description  : product description
        item_category     : product category
        pro_model         : Gemini Pro instance
        index             : FAISS index
        metadata          : parallel metadata list
        embedding_model   : SentenceTransformer instance
        nigerian_language : optional cultural conditioning
        top_k             : number of examples to retrieve
        max_retries       : retry attempts on failure
    """

    # ── RETRIEVE RELEVANT EXAMPLES ────────────────────────────────
    retrieved = retrieve_for_agent2(
        user_id             = dossier.user_id,
        item_title          = item_title,
        item_description    = item_description,
        item_category       = item_category,
        index               = index,
        metadata            = metadata,
        embedding_model     = embedding_model,
        top_k               = top_k,
        same_category_boost = True
    )

    rag_block    = format_rag_examples(retrieved)
    n_same_cat   = sum(1 for r in retrieved if r.get('same_category'))

    # ── RATING ANCHOR ─────────────────────────────────────────────
    low  = max(1, round(dossier.avg_rating - dossier.rating_std))
    high = min(5, round(dossier.avg_rating + dossier.rating_std))

    # ── NIGERIAN CONDITIONING ─────────────────────────────────────
    lang_instructions = {
        'pidgin'           : "Nigerian Pidgin — weave in naturally: 'e good o', 'I no go lie', 'wahala', 'sha'",
        'yoruba_influenced': "Yoruba-influenced English — 'omo', 'my people', occasional Yoruba words",
        'igbo_influenced'  : "Igbo-influenced English — 'nna', 'chai', direct assertive tone",
        'hausa_influenced' : "Hausa-influenced English — 'wallahi', 'alhamdulillah', respectful tone",
    }
    lang_line = ""
    if nigerian_language and nigerian_language in lang_instructions:
        lang_line = f"\nLANGUAGE STYLE: {lang_instructions[nigerian_language]}\n"

    # ── CONCRETE OUTPUT EXAMPLE ───────────────────────────────────
    output_example = '''{
  "reasoning": "This user writes practical step-by-step reviews and always mentions value. The product is mid-range which fits their purchase pattern.",
  "rating": 4,
  "title": "Works as described",
  "review": "Easy to set up. Does exactly what it claims. Price is fair for what you get. Been using for two weeks with no issues.",
  "user_id": "USER123",
  "asin": "B00EXAMPLE",
  "language": "english"
}'''

    # ── AGENT 2 PROMPT WITH RAG ───────────────────────────────────
    prompt = f"""You are simulating a specific Amazon reviewer writing about a product they have never reviewed before.
You have two sources of truth. Use both.

══════════════════════════════════════════════
SOURCE 1 — ANALYST BRIEF (who this person is)
══════════════════════════════════════════════
{dossier.simulation_brief}

Identity : {dossier.core_identity}
Voice    : {dossier.writing_voice.tone} | {dossier.writing_voice.style} | {dossier.writing_voice.structure}
Phrases  : {', '.join(dossier.writing_voice.signature_phrases) or 'none identified'}
Rating   : {dossier.rating_behaviour.tendency} rater | anchor {low}–{high} stars
Never gives: {dossier.rating_behaviour.never_gives or 'no restrictions'}
Cares about: {', '.join(dossier.what_they_care_about[:4])}
Ignores    : {', '.join(dossier.what_they_ignore[:3]) if dossier.what_they_ignore else 'nothing notable'}

Traits:
{chr(10).join(f'  • {t}' for t in dossier.deep_traits)}

══════════════════════════════════════════════
SOURCE 2 — THEIR ACTUAL WRITING ({n_same_cat} from same category)
══════════════════════════════════════════════
Study these carefully. This is how this person actually writes.
Match their sentence length, vocabulary, structure, and emotional register exactly.

{rag_block}
{lang_line}
══════════════════════════════════════════════
PRODUCT TO REVIEW
══════════════════════════════════════════════
Title      : {item_title}
ASIN       : {item_asin}
Category   : {item_category}
Description: {str(item_description)[:400]}

══════════════════════════════════════════════
ANTI-DRIFT RULES
══════════════════════════════════════════════
- Read SOURCE 2 carefully before writing anything
- Do NOT default to generic enthusiastic review language
- Do NOT use words like "fantastic", "amazing", "must-have" unless they appear in SOURCE 2
- Do NOT open with "I am so happy" or emotional declarations unless SOURCE 2 shows this pattern
- Match the sentence length visible in SOURCE 2
- If SOURCE 2 shows terse practical reviews — write tersely and practically
- The rating must be coherent with the review text — decide the rating first

OUTPUT RULES:
- Return ONLY a single JSON object
- No text before the opening brace
- No text after the closing brace  
- No markdown fences
- rating field must appear before review field

EXAMPLE OF CORRECT FORMAT:
{output_example}

Set user_id="{dossier.user_id}", asin="{item_asin}", language="{nigerian_language or 'english'}"
"""

    # ── GENERATION WITH RETRY ─────────────────────────────────────
    for attempt in range(max_retries + 1):
        try:
            response = pro_model.generate_content(prompt)
            raw      = response.text.strip()

            # clean fences
            raw = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
            raw = re.sub(r'^```\s*',     '', raw, flags=re.MULTILINE)
            raw = re.sub(r'\s*```$',     '', raw, flags=re.MULTILINE)
            raw = raw.strip()

            # extract first complete JSON object only
            brace_count = 0
            end_pos     = 0
            for i, char in enumerate(raw):
                if char == '{':
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0:
                        end_pos = i + 1
                        break
            if end_pos > 0:
                raw = raw[:end_pos]

            data = json.loads(raw)

            # catch meta-response
            if 'description' in data and 'review' not in data:
                raise ValueError(f"Meta-response detected. Keys: {list(data.keys())}")

            # force identifiers
            data['user_id']  = dossier.user_id
            data['asin']     = item_asin
            data['language'] = nigerian_language or 'english'

            review = SimulatedReview(**data)
            return review

        except json.JSONDecodeError as e:
            print(f"  ⚠️  RAG Agent 2 attempt {attempt+1} — JSON: {e}")
        except ValueError as e:
            print(f"  ⚠️  RAG Agent 2 attempt {attempt+1} — Validation: {e}")
        except Exception as e:
            print(f"  ⚠️  RAG Agent 2 attempt {attempt+1} — Error: {e}")

    # ── FALLBACK ──────────────────────────────────────────────────
    print(f"  ❌ RAG Agent 2 failed for {dossier.user_id} — fallback")
    return SimulatedReview(
        user_id   = dossier.user_id,
        asin      = item_asin,
        reasoning = "Fallback after RAG generation failure.",
        rating    = round(dossier.avg_rating),
        title     = "Product Review",
        review    = (
            "This product meets expectations. "
            f"{'Good value overall.' if dossier.avg_rating >= 3.5 else 'Did not fully meet expectations.'}"
        ),
        language  = nigerian_language or 'english'
    )




In [39]:
# ── MASTER PIPELINE ───────────────────────────────────────────────

def simulate_review_two_agent(
    user_id          : str,
    texts            : list,
    ratings          : list,
    item_asin        : str,
    item_title       : str,
    item_description : str,
    item_category    : str,          # ← added — was missing, caused holdout bug
    flash_model,
    pro_model,
    rag_index,                       # ← added — no longer hardcoded global
    rag_metadata     : list,         # ← added
    embedding_model,                 # ← added
    nigerian_language: Optional[str] = None,
    top_k            : int           = 4,
    verbose          : bool          = False
) -> tuple[PersonaDossier, SimulatedReview]:
    """
    Full two-agent pipeline with RAG grounding.

    Agent 1 (Flash)  — analyses history, builds dossier, cached
    Agent 2 (Pro)    — generates review from dossier + RAG examples

    Parameters:
        user_id           : reviewer ID
        texts             : list of their past review texts
        ratings           : list of their past ratings (same order)
        item_asin         : product ASIN
        item_title        : product title
        item_description  : product description
        item_category     : product category — used for RAG retrieval boost
        flash_model       : Gemini Flash instance (Agent 1)
        pro_model         : Gemini Pro instance (Agent 2)
        rag_index         : loaded FAISS index
        rag_metadata      : parallel metadata list
        embedding_model   : loaded SentenceTransformer
        nigerian_language : optional cultural conditioning
        top_k             : number of RAG examples to retrieve
        verbose           : print step-by-step progress
    """

    # ── AGENT 1 ──────────────────────────────────────────────────
    if verbose:
        hit = cache.exists(user_id)
        print(f"\n[Agent 1] {'Cache hit ✅' if hit else 'Analysing with Flash...'}")

    dossier = run_agent1_analyst(
        user_id     = user_id,
        texts       = texts,
        ratings     = ratings,
        flash_model = flash_model
    )

    if verbose:
        print(f"  Traits found : {len(dossier.deep_traits)}")
        print(f"  Identity     : {dossier.core_identity[:70]}...")
        print(f"[Agent 2] Generating with RAG + Pro...")

    # ── AGENT 2 WITH RAG ─────────────────────────────────────────
    review = run_agent2_with_rag(
        dossier           = dossier,
        item_asin         = item_asin,
        item_title        = item_title,
        item_description  = item_description,
        item_category     = item_category,    # ← clean — no holdout reference
        pro_model         = pro_model,
        index             = rag_index,
        metadata          = rag_metadata,
        embedding_model   = embedding_model,
        nigerian_language = nigerian_language,
        top_k             = top_k
    )

    if verbose:
        print(f"  Rating       : {review.rating}★")
        print(f"  Language     : {review.language}")

    return dossier, review

### RAG Implementation 
cell above cannot run without the rag cell so run it first or change the arriangement

In [40]:
#this function is needed below
def safe_timestamp(value) -> float:
    """
    Converts any timestamp format to a float unix timestamp.
    Handles: int, float, pd.Timestamp, datetime, string, None
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return 0.0
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, pd.Timestamp):
        return float(value.timestamp())
    if hasattr(value, 'timestamp'):          # any datetime-like object
        return float(value.timestamp())
    try:
        return float(pd.Timestamp(value).timestamp())
    except Exception:
        return 0.0



# ── CONSTANTS ────────────────────────────────────────────────────
RAG_INDEX_PATH   = '/kaggle/working/rag_index.faiss'
RAG_META_PATH    = '/kaggle/working/rag_meta.pkl'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'   # fast, lightweight, strong for semantic similarity


# ── STEP 1: BUILD THE RAG INDEX ───────────────────────────────────

def build_rag_index(persona_df: pd.DataFrame, embedding_model: SentenceTransformer):
    """
    Builds a FAISS vector index over all reviews in persona_df.
    Run once during setup — saved to disk and reloaded on demand.

    Each review is stored with its:
    - embedding vector  (for similarity search)
    - user_id           (so we only retrieve from the right user)
    - review text       (what gets passed to Agent 2)
    - rating            (for context)
    - category          (for category-aware retrieval)

    Parameters:
        persona_df      : your persona set from split_user_reviews()
        embedding_model : loaded SentenceTransformer instance

    Returns:
        index    : FAISS index
        metadata : list of dicts, one per review — parallel to index
    """

    print("Building RAG index...")
    print(f"  Reviews to index: {len(persona_df):,}")

    texts    = persona_df['text'].fillna('').tolist()
    metadata = []

    for _, row in persona_df.iterrows():
        metadata.append({
            'user_id'   : row.get('user_id', ''),
            'text'      : str(row.get('text', '')),
            'rating'    : float(row.get('rating', 3.0)),
            'category'  : str(row.get('main_category', '')),
            'asin'      : str(row.get('parent_asin', '')),
            'timestamp' : safe_timestamp(row.get('timestamp', 0)),
        })

    # encode all reviews in batches — memory efficient
    print("  Encoding reviews...")
    embeddings = embedding_model.encode(
        texts,
        batch_size      = 64,
        show_progress_bar = True,
        convert_to_numpy  = True,
        normalize_embeddings = True    # normalise for cosine similarity
    )

    # build flat FAISS index — exact search, good for this dataset size
    dimension = embeddings.shape[1]
    index     = faiss.IndexFlatIP(dimension)   # inner product = cosine on normalised vectors
    index.add(embeddings.astype(np.float32))

    # save to disk
    faiss.write_index(index, RAG_INDEX_PATH)
    with open(RAG_META_PATH, 'wb') as f:
        pickle.dump(metadata, f)

    print(f"  Index built: {index.ntotal:,} vectors, dimension {dimension}")
    print(f"  Saved to {RAG_INDEX_PATH}")

    return index, metadata


# ── STEP 2: LOAD THE RAG INDEX ────────────────────────────────────

def load_rag_index():
    """
    Loads the FAISS index and metadata from disk.
    Call this at evaluation time instead of rebuilding.
    """
    if not os.path.exists(RAG_INDEX_PATH):
        raise FileNotFoundError(
            f"RAG index not found at {RAG_INDEX_PATH}. "
            f"Run build_rag_index() first."
        )

    index = faiss.read_index(RAG_INDEX_PATH)
    with open(RAG_META_PATH, 'rb') as f:
        metadata = pickle.load(f)

    print(f"RAG index loaded: {index.ntotal:,} vectors")
    return index, metadata


# ── STEP 3: RETRIEVE FOR AGENT 2 ─────────────────────────────────

def retrieve_for_agent2(
    user_id             : str,
    item_title          : str,
    item_description    : str,
    item_category       : str,
    index               : faiss.Index,
    metadata            : list,
    embedding_model     : SentenceTransformer,
    top_k               : int  = 4,
    same_category_boost : bool = True
) -> list[dict]:

    # ── BUILD PRODUCT QUERY ───────────────────────────────────────
    query_parts = [p for p in [item_title, item_description, item_category] if p]
    query       = ' '.join(query_parts)[:512]

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True
    ).astype(np.float32)

    # ── SEARCH FULL INDEX ─────────────────────────────────────────
    search_k        = index.ntotal   # always search everything
    scores, indices = index.search(query_embedding, search_k)
    scores          = scores[0]
    indices         = indices[0]

    # ── FILTER TO THIS USER ───────────────────────────────────────
    user_results = []

    for score, idx in zip(scores, indices):
        if idx < 0 or idx >= len(metadata):
            continue

        record = metadata[idx]

        if record['user_id'] != user_id:
            continue

        if len(record['text'].split()) < 10:
            continue

        user_results.append({
            'text'            : record['text'],
            'rating'          : record['rating'],
            'category'        : record['category'],
            'asin'            : record['asin'],
            'timestamp'       : record.get('timestamp', 0),
            'similarity_score': float(score),
            'same_category'   : (
                record['category'].lower() == item_category.lower()
                if item_category else False
            ),
            'source'          : 'rag'
        })

    # ── CATEGORY BOOST ────────────────────────────────────────────
    if same_category_boost:
        for r in user_results:
            if r['same_category']:
                r['similarity_score'] += 0.15

    # ── RANK ──────────────────────────────────────────────────────
    user_results.sort(key=lambda x: x['similarity_score'], reverse=True)

    # ── CHECK FOR SAME-CATEGORY RESULTS ──────────────────────────
    # if top results have no same-category match at all,
    # trigger fallback to latest reviews instead
    same_cat_found = any(r['same_category'] for r in user_results[:top_k])

    if user_results and same_cat_found:
        # happy path — semantic results include category match
        return user_results[:top_k]

    if user_results and not same_cat_found:
        # we have results but none match the category
        # return what we have — better than nothing
        # but log it so you know
        print(
            f"  ℹ️  No same-category results for {user_id} "
            f"in '{item_category}' — returning best semantic matches"
        )
        return user_results[:top_k]

    # ── FALLBACK: no results at all — return latest reviews ───────
    print(f"  ⚠️  No RAG results for {user_id} — falling back to latest reviews")

    all_user_records = [
        {
            'text'            : m['text'],
            'rating'          : m['rating'],
            'category'        : m['category'],
            'asin'            : m['asin'],
            'timestamp'       : m.get('timestamp', 0),
            'similarity_score': 0.0,
            'same_category'   : (
                m['category'].lower() == item_category.lower()
                if item_category else False
            ),
            'source'          : 'fallback_latest'
        }
        for m in metadata
        if m['user_id'] == user_id
        and len(m['text'].split()) >= 10
    ]

    if not all_user_records:
        print(f"  ❌ User {user_id} has no usable records in metadata at all")
        return []

    # sort by timestamp — most recent first
    all_user_records.sort(key=lambda x: x['timestamp'], reverse=True)
    top_results = all_user_records[:top_k]
    print(f"  ✅ Fallback returned {len(top_results)} latest reviews")
    return top_results

# ── STEP 4: FORMAT FOR AGENT 2 PROMPT ────────────────────────────

def format_rag_examples(retrieved_reviews: list[dict]) -> str:
    """
    Formats retrieved reviews into a clean block for Agent 2's prompt.
    Shows rating + text so Agent 2 sees both the voice and the rating pattern.
    """

    if not retrieved_reviews:
        return "No similar past reviews found for this user."

    lines = []
    for i, review in enumerate(retrieved_reviews):
        category_note = " [same category]" if review.get('same_category') else ""
        lines.append(
            f"Past review {i+1}{category_note} — {int(review['rating'])}★:\n"
            f"\"{review['text'].strip()}\""
        )

    return "\n\n".join(lines)


In [41]:
# ── SETUP AND TEST ────────────────────────────────────────────────

# load embedding model once — reuse everywhere
print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("✅ Embedding model loaded")

# build index once — reuse across all evaluations
if os.path.exists(RAG_INDEX_PATH):
    print("Loading existing RAG index...")
    rag_index, rag_metadata = load_rag_index()
else:
    print("Building RAG index from persona_df...")
    rag_index, rag_metadata = build_rag_index(persona_df, embedder)

# ── QUICK RETRIEVAL TEST ──────────────────────────────────────────

test_user_id = persona_df['user_id'].iloc[0]

retrieved = retrieve_for_agent2(
    user_id          = "AE7P3G7DWP3VVFKOK2H2PPTK2TOA", #test_user_id,
    item_title       = "Electric Shaver for Women ",
    item_description = " ",
    item_category    = "All Beauty",
    index            = rag_index,
    metadata         = rag_metadata,
    embedding_model  = embedder,
    top_k            = 4
)

print(f"\nRetrieved {len(retrieved)} reviews for test user")
for r in retrieved:
    same = "✅ same category" if r['same_category'] else ""
    print(f"  {r['rating']}★ | sim: {r['similarity_score']:.3f} | {same}")
    print(f"  {r['text'][:100]}...")
    print()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded
Loading existing RAG index...
RAG index loaded: 86 vectors

Retrieved 4 reviews for test user
  5.0★ | sim: 0.478 | ✅ same category
  I purchased these body wax strips and began using them only a few days after letting my hair grow ou...

  4.0★ | sim: 0.447 | ✅ same category
  This clipper has several different heads that are easy to switch out.  The heads can be switched out...

  5.0★ | sim: 0.445 | ✅ same category
  I have been pleasantly surprised by this product.  I have been an esthetician for 2 decades now and ...

  4.0★ | sim: 0.350 | ✅ same category
  I was surprised by how lightweight this primer is.  The texture reminds me of a light moisturizer wh...



In [42]:
def inspect_rag_index(metadata: list, n: int = 5):
    """
    Inspect the first n records in the RAG metadata.
    Shows exactly what was stored during index build.
    """
    print(f"Total records in metadata: {len(metadata)}")
    print(f"\nFirst {n} records:")
    print("=" * 60)
    
    for i, record in enumerate(metadata[:n]):
        print(f"\nRecord {i+1}:")
        for key, value in record.items():
            display = repr(value)[:80] if isinstance(value, str) else value
            print(f"  {key:<15} : {display}")
        print(f"  text word count : {len(record['text'].split())}")
    
    print("\n" + "=" * 60)
    print("TEXT POPULATION CHECK:")
    empty_text  = sum(1 for m in metadata if not m['text'].strip())
    filled_text = sum(1 for m in metadata if m['text'].strip())
    print(f"  Records with text    : {filled_text}")
    print(f"  Records without text : {empty_text}")

inspect_rag_index(rag_metadata)

Total records in metadata: 86

First 5 records:

Record 1:
  user_id         : 'AE7P3G7DWP3VVFKOK2H2PPTK2TOA'
  text            : "This makeup is crazy.  It comes out of the bottle looking like a white colored 
  rating          : 4.0
  category        : 'All Beauty'
  asin            : 'B07VNQ4G13'
  timestamp       : 1567113098.664
  text word count : 158

Record 2:
  user_id         : 'AE7P3G7DWP3VVFKOK2H2PPTK2TOA'
  text            : 'This foundation is like no other foundation.  It is definitely not your average
  rating          : 5.0
  category        : 'All Beauty'
  asin            : 'B07VML1QZC'
  timestamp       : 1569596918.78
  text word count : 97

Record 3:
  user_id         : 'AE7P3G7DWP3VVFKOK2H2PPTK2TOA'
  text            : "This under eye cream is amazing.  The first time I used it, I could immediately
  rating          : 5.0
  category        : 'All Beauty'
  asin            : 'B07X1PH59J'
  timestamp       : 1570706272.243
  text word count : 137

Record 4:
  user_

In [48]:
# ── TEST ─────────────────────────────────────────────────────────

test_user_id = persona_df['user_id'].value_counts().index[0]
test_reviews = persona_df[persona_df['user_id'] == test_user_id]

dossier, simulated = simulate_review_two_agent(
    user_id           = persona_df['user_id'],
    texts             = persona_df['text'],
    ratings           = persona_df['rating'],
    item_asin         = persona_df['parent_asin'],
    item_title        = persona_df['product_title'],
    item_description  = persona_df['description_text'],
    item_category     = holdout.get('main_category', ''),  # holdout is defined here
    flash_model       = flash_model,
    pro_model         = pro_model,
    rag_index         = rag_index,
    rag_metadata      = rag_metadata,
    embedding_model   = embedding_model,
    nigerian_language = nigerian_language,
    verbose           = False
)

# both are typed objects — access with dot notation, no dict guessing
print(f"\nDOSSIER")
print(f"  Identity : {dossier.core_identity}")
print(f"  Traits   : {dossier.deep_traits}")
print(f"  Brief    : {dossier.simulation_brief}")

print(f"\nREVIEW")
print(f"  Reasoning: {review.reasoning}")
print(f"  Rating   : {review.rating}★")
print(f"  Title    : {review.title}")
print(f"  Review   :\n{review.review}")

NameError: name 'holdout' is not defined

In [47]:
persona_df.head()

,rating,review_title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,description_text,features_text,price,store,main_category,review_seq,word_count,split
0,4,Doesn't leave marks on clothes!,This makeup is crazy. It comes out of the bot...,B07VNQ4G13,B07VNQ4G13,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-08-29 21:11:38.664,0,True,"Flawless Finish Foundation, Colour Changing Fo...",,,NaN,CIDBEST,All Beauty,1,158,persona
1,5,Colour Changing Foundation,This foundation is like no other foundation. ...,B07VML1QZC,B07VML1QZC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-09-27 15:08:38.780,0,True,"Flawless Liquid Foundation Cream, Liquid Found...",,,NaN,Cherioll,All Beauty,2,97,persona
2,5,Immediately see the difference...,This under eye cream is amazing. The first ti...,B07X1PH59J,B07X1PH59J,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-10 11:17:52.243,0,True,"Rapid Reduction Eye Cream, Under Eye Cream, Un...",,,NaN,CIDBEST,All Beauty,3,137,persona
3,4,Makeup concealer,"This 3 pack of cream concealer is well, very c...",B07XNYVBY5,B07XNYVBY5,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-10-31 07:16:34.158,0,True,"Concealer Cream,Makeup concealer,Skin Lighteni...",,,NaN,SCOBUTY,All Beauty,4,132,persona
4,4,Lightweight Primer,I was surprised by how lightweight this primer...,B0828LCHWC,B0828LCHWC,AE7P3G7DWP3VVFKOK2H2PPTK2TOA,2019-12-23 06:14:41.575,0,True,W-Airfit Primer Face Makeup Base Pink Isolatio...,,,NaN,Lofu,All Beauty,5,124,persona


## Evaluation function 
- Evaluation using Review Text Quality (ROUGE /BERTScore) & Rating Accuracy (RMSE)

In [49]:

# ── SET HUGGINGFACE TOKEN ────────────────────────────────────────
HF_TOKEN = HF_Key
os.environ["HF_TOKEN"] = HF_TOKEN


# ── ROUGE ────────────────────────────────────────────────────────

def compute_rouge(generated: str, reference: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer = True
    )
    scores = scorer.score(reference, generated)
    return {
        'rouge1' : round(scores['rouge1'].fmeasure, 4),
        'rouge2' : round(scores['rouge2'].fmeasure, 4),
        'rougeL' : round(scores['rougeL'].fmeasure, 4),
    }


# ── BERTSCORE ────────────────────────────────────────────────────

def compute_bertscore(generated_list: list, reference_list: list) -> list:
    P, R, F1 = bert_score(
        generated_list,
        reference_list,
        lang       = "en",
        model_type = "roberta-large",
        verbose    = False
    )
    return [round(f.item(), 4) for f in F1]


# ── RMSE ─────────────────────────────────────────────────────────

def compute_rmse(true_ratings: list, pred_ratings: list) -> float:
    true = np.array(true_ratings, dtype=float)
    pred = np.array(pred_ratings, dtype=float)
    return round(float(np.sqrt(np.mean((true - pred) ** 2))), 4)


# ── EVALUATION PIPELINE ──────────────────────────────────────────

def evaluate_pipeline(
    persona_df        : pd.DataFrame,
    val_df            : pd.DataFrame,
    flash_model,
    pro_model,
    rag_index,                            # ← added explicitly
    rag_metadata      : list,             # ← added explicitly
    embedding_model,                      # ← added explicitly
    nigerian_language : str  = None,
    n_users           : int  = 20,
    verbose           : bool = True,
    mode              : str  = 'validation'
):
    if mode == 'test':
        print("⚠️  WARNING: You are evaluating on the TEST SET.")
        print("   Only do this once — at final submission.")
        print("   If you are still tuning, use mode='validation'\n")

    results = []
    skipped = 0

    print("=" * 60)
    print(f"EVALUATION PIPELINE  [{mode.upper()}]")
    print("=" * 60)
    print(f"  Users to evaluate : {n_users}")
    print(f"  Language mode     : {nigerian_language or 'english'}")
    print(f"  Agent 1           : Gemini Flash (persona)")
    print(f"  Agent 2           : Gemini Pro + RAG (generation)")
    print()

    # ── OVERLAP CHECK ─────────────────────────────────────────────
    persona_users = set(persona_df['user_id'].unique())
    val_users     = set(val_df['user_id'].unique())
    eval_users    = list(persona_users & val_users)[:n_users]

    if len(eval_users) == 0:
        print("❌ No overlapping users between persona_df and val_df")
        print(f"   Persona users : {len(persona_users)}")
        print(f"   Val users     : {len(val_users)}")
        return pd.DataFrame(), {}

    print(f"  Eligible users    : {len(eval_users)}")
    print()

    # ── PER USER LOOP ─────────────────────────────────────────────
    for idx, user_id in enumerate(eval_users):

        # ── PERSONA HISTORY ──────────────────────────────────────
        history = persona_df[
            persona_df['user_id'] == user_id
        ].sort_values('timestamp')

        texts   = history['text'].tolist()
        ratings = history['rating'].tolist()

        if len(texts) < 3:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... skipped — insufficient history")
            skipped += 1
            continue

        # ── GROUND TRUTH ─────────────────────────────────────────
        holdout_rows = val_df[val_df['user_id'] == user_id]
        if len(holdout_rows) == 0:
            skipped += 1
            continue

        holdout = holdout_rows.iloc[0]

        # ── FIX: safe pandas Series access ───────────────────────
        def safe_get(row, col, default=''):
            val = row[col] if col in row.index else default
            return default if pd.isna(val) else val

        item_asin        = safe_get(holdout, 'parent_asin', 'UNKNOWN')
        item_title       = safe_get(holdout, 'product_title', f'Product {item_asin}')
        item_description = safe_get(holdout, 'description_text', '')
        item_category    = safe_get(holdout, 'main_category', '')
        true_review      = str(safe_get(holdout, 'text', ''))
        true_rating_raw  = safe_get(holdout, 'rating', 3)

        # never leak true review text as description
        if not item_description:
            item_description = f"Amazon product: {item_title}"

        try:
            true_rating = int(float(true_rating_raw))
        except (ValueError, TypeError):
            true_rating = 3

        if not true_review or len(true_review.split()) < 5:
            skipped += 1
            continue

        # ── TWO-AGENT SIMULATION ──────────────────────────────────
        try:
            dossier, simulated = simulate_review_two_agent(
                user_id           = user_id,
                texts             = texts,
                ratings           = ratings,
                item_asin         = item_asin,
                item_title        = item_title,
                item_description  = item_description,
                item_category     = item_category,
                flash_model       = flash_model,
                pro_model         = pro_model,
                rag_index         = rag_index,
                rag_metadata      = rag_metadata,
                embedding_model   = embedding_model,
                nigerian_language = nigerian_language,
                verbose           = False
            )

            generated_review = simulated.review
            pred_rating      = simulated.rating
            reasoning        = simulated.reasoning

        except Exception as e:
            if verbose:
                print(f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... ⚠️  failed: {e}")
            skipped += 1
            continue

        if not generated_review or len(generated_review.strip()) < 10:
            skipped += 1
            continue

        # ── ROUGE ─────────────────────────────────────────────────
        rouge_scores = compute_rouge(
            generated = generated_review,
            reference = true_review
        )

        # ── TRAIT CONSISTENCY ─────────────────────────────────────
        trait_hits = 0
        if dossier.deep_traits:
            gen_lower = generated_review.lower()
            for trait in dossier.deep_traits:
                trait_words = [
                    w for w in trait.lower().split()
                    if len(w) > 4
                ]
                if any(w in gen_lower for w in trait_words):
                    trait_hits += 1
            trait_consistency = round(trait_hits / len(dossier.deep_traits), 3)
        else:
            trait_consistency = 0.0

        results.append({
            'user_id'          : user_id,
            'true_rating'      : true_rating,
            'pred_rating'      : pred_rating,
            'rating_error'     : abs(true_rating - pred_rating),
            'true_review'      : true_review,
            'generated_review' : generated_review,
            'reasoning'        : reasoning,
            'rouge1'           : rouge_scores['rouge1'],
            'rouge2'           : rouge_scores['rouge2'],
            'rougeL'           : rouge_scores['rougeL'],
            'bertscore'        : None,
            'trait_consistency': trait_consistency,
            'n_traits_found'   : len(dossier.deep_traits),
            'persona_type'     : dossier.rating_behaviour.tendency.value,
            'writing_style'    : dossier.writing_voice.style.value,
            'simulation_brief' : dossier.simulation_brief,
        })

        if verbose:
            print(
                f"[{idx+1}/{len(eval_users)}] {user_id[:14]}... "
                f"| True: {true_rating}★  Pred: {pred_rating}★ "
                f"| ROUGE-1: {rouge_scores['rouge1']:.3f} "
                f"| Traits hit: {trait_hits}/{len(dossier.deep_traits)}"
            )

    # ── BERTSCORE IN BATCH ────────────────────────────────────────
    if results:
        print(f"\nRunning BERTScore on {len(results)} pairs...")
        try:
            bert_scores = compute_bertscore(
                generated_list = [r['generated_review'] for r in results],
                reference_list = [r['true_review']      for r in results]
            )
            for i, score in enumerate(bert_scores):
                results[i]['bertscore'] = score
        except Exception as e:
            print(f"⚠️  BERTScore failed: {e}")

    # ── BUILD RESULTS DATAFRAME ───────────────────────────────────
    results_df = pd.DataFrame(results)

    if len(results_df) == 0:
        print("❌ No results generated")
        return results_df, {}

    # ── FIX: safe bertscore fill ──────────────────────────────────
    bert_mean = results_df['bertscore'].mean()
    if pd.isna(bert_mean):
        print("⚠️  All BERTScores are None — check roberta-large load")
        results_df['bertscore'] = 0.0
    else:
        results_df['bertscore'] = results_df['bertscore'].fillna(bert_mean)

    # ── SUMMARY ───────────────────────────────────────────────────
    summary = {
        'mode'                 : mode,
        'n_evaluated'          : len(results_df),
        'n_skipped'            : skipped,
        'rmse'                 : compute_rmse(
                                     results_df['true_rating'].tolist(),
                                     results_df['pred_rating'].tolist()
                                 ),
        'mae'                  : round(results_df['rating_error'].mean(), 4),
        'avg_rouge1'           : round(results_df['rouge1'].mean(),           4),
        'avg_rouge2'           : round(results_df['rouge2'].mean(),           4),
        'avg_rougeL'           : round(results_df['rougeL'].mean(),           4),
        'avg_bertscore'        : round(results_df['bertscore'].mean(),        4),
        'avg_trait_consistency': round(results_df['trait_consistency'].mean(), 4),
    }

    print(f"\n{'=' * 60}")
    print(f"RESULTS  [{mode.upper()}]")
    print(f"{'=' * 60}")
    print(f"  Users evaluated      : {summary['n_evaluated']}")
    print(f"  Users skipped        : {summary['n_skipped']}")
    print(f"\n  ── RATING ──────────────────────────────")
    print(f"  RMSE                 : {summary['rmse']}")
    print(f"  MAE                  : {summary['mae']}")
    print(f"  (target RMSE < 1.0  |  strong < 0.8)")
    print(f"\n  ── TEXT QUALITY ────────────────────────")
    print(f"  Avg ROUGE-1          : {summary['avg_rouge1']}")
    print(f"  Avg ROUGE-2          : {summary['avg_rouge2']}")
    print(f"  Avg ROUGE-L          : {summary['avg_rougeL']}")
    print(f"  Avg BERTScore        : {summary['avg_bertscore']}")
    print(f"  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)")
    print(f"\n  ── BEHAVIOURAL FIDELITY ────────────────")
    print(f"  Avg trait consistency: {summary['avg_trait_consistency']}")
    print(f"  (proxy — % of persona traits reflected in generated review)")
    print(f"{'=' * 60}\n")

    print("TOP 3 — highest BERTScore:")
    best = results_df.nlargest(3, 'bertscore')
    for _, row in best.iterrows():
        print(f"  {row['user_id'][:16]} | BERTScore: {row['bertscore']} | ROUGE-1: {row['rouge1']}")

    print("\nBOTTOM 3 — lowest BERTScore (investigate these):")
    worst = results_df.nsmallest(3, 'bertscore')
    for _, row in worst.iterrows():
        print(f"\n  User     : {row['user_id']}")
        print(f"  True     : {row['true_review'][:150]}...")
        print(f"  Generated: {row['generated_review'][:150]}...")
        print(f"  Brief    : {row['simulation_brief'][:120]}...")
        print(f"  BERTScore: {row['bertscore']}  ROUGE-1: {row['rouge1']}")

    return results_df, summary


# ── RUN ───────────────────────────────────────────────────────────

results_df, summary = evaluate_pipeline(
    persona_df        = persona_df,
    val_df            = val_df,
    flash_model       = flash_model,
    pro_model         = pro_model,
    rag_index         = rag_index,
    rag_metadata      = rag_metadata,
    embedding_model   = embedder,
    nigerian_language = None,
    n_users           = 20,
    verbose           = True,
    mode              = 'validation'
)

results_df.to_csv('/kaggle/working/validation_results.csv', index=False)
print(f"Results saved to /kaggle/working/validation_results.csv")


# ── FINAL TEST EVAL — run this ONCE at submission ────────────────
# uncomment only when done tuning

# dossier, review = simulate_review_two_agent(
#     user_id           = test_user_id,
#     texts             = test_reviews['text'].tolist(),
#     ratings           = test_reviews['rating'].tolist(),
#     item_asin         = test_reviews.iloc[-1]['parent_asin'],
#     item_title        = test_reviews.iloc[-1].get('product_title', 'Beauty Product'),
#     item_description  = test_reviews.iloc[-1].get('description_text', ''),
#     item_category     = test_reviews.iloc[-1].get('main_category', ''),
#     flash_model       = flash_model,
#     pro_model         = pro_model,
#     rag_index         = rag_index,
#     rag_metadata      = rag_metadata,
#     embedding_model   = embedder,
#     verbose           = True
# )
# test_results_df.to_csv('/kaggle/working/test_results_final.csv', index=False)

EVALUATION PIPELINE  [VALIDATION]
  Users to evaluate : 20
  Language mode     : english
  Agent 1           : Gemini Flash (persona)
  Agent 2           : Gemini Pro + RAG (generation)

  Eligible users    : 7

[1/7] AFSCJNRG4BAGAB... | True: 5★  Pred: 5★ | ROUGE-1: 0.361 | Traits hit: 2/7
[2/7] AFUBMCVI5J6G4F... | True: 4★  Pred: 3★ | ROUGE-1: 0.296 | Traits hit: 2/5
[3/7] AGUTZC4GHLTGYH... | True: 5★  Pred: 5★ | ROUGE-1: 0.267 | Traits hit: 0/5
[4/7] AE7P3G7DWP3VVF... | True: 5★  Pred: 4★ | ROUGE-1: 0.351 | Traits hit: 3/5
[5/7] AGFN3252BBTYUJ... | True: 5★  Pred: 5★ | ROUGE-1: 0.286 | Traits hit: 1/4
[6/7] AFR4BTNWATNG7O... | True: 5★  Pred: 5★ | ROUGE-1: 0.277 | Traits hit: 4/5
[7/7] AEYKTZXAWOPJG5... | True: 5★  Pred: 5★ | ROUGE-1: 0.400 | Traits hit: 4/5

Running BERTScore on 7 pairs...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



RESULTS  [VALIDATION]
  Users evaluated      : 7
  Users skipped        : 0

  ── RATING ──────────────────────────────
  RMSE                 : 0.5345
  MAE                  : 0.2857
  (target RMSE < 1.0  |  strong < 0.8)

  ── TEXT QUALITY ────────────────────────
  Avg ROUGE-1          : 0.3197
  Avg ROUGE-2          : 0.0567
  Avg ROUGE-L          : 0.1762
  Avg BERTScore        : 0.8464
  (ROUGE-1 > 0.3 reasonable  |  BERTScore > 0.85 strong)

  ── BEHAVIOURAL FIDELITY ────────────────
  Avg trait consistency: 0.448
  (proxy — % of persona traits reflected in generated review)

TOP 3 — highest BERTScore:
  AEYKTZXAWOPJG5MG | BERTScore: 0.8806 | ROUGE-1: 0.4
  AFSCJNRG4BAGAB37 | BERTScore: 0.8671 | ROUGE-1: 0.3614
  AE7P3G7DWP3VVFKO | BERTScore: 0.8447 | ROUGE-1: 0.351

BOTTOM 3 — lowest BERTScore (investigate these):

  User     : AGFN3252BBTYUJUUDQMAGYZNUT5A_2_1
  True     : [[VIDEOID:8382b0ff456e99e9dca6aaf75724d67e]] Safety protective mask, simple color design, thick season is

In [ ]:
persona_df.shape